In [0]:
#display the data from the table employee
display(spark.read.table("plworkforce_catalog.001_bronze.employee"))

from pyspark.sql import functions as F

# Use silver schema
spark.sql("USE plworkforce_catalog.002_silver")

# Read from Bronze
df = spark.table("plworkforce_catalog.001_bronze.employee")

# Transform
df_clean = (df

    # Clean text
    .withColumn("first_name", F.initcap(F.trim("first_name")))
    .withColumn("last_name", F.initcap(F.trim("last_name")))
    .withColumn("email", F.lower(F.trim("email")))

    # Create derived column (IMPORTANT)
    .withColumn("full_name",
        F.concat_ws(" ", F.col("first_name"), F.col("last_name"))
    )
    
    #convert dates
    .withColumn("hire_date", F.to_date("hire_date", "dd-MM-yyyy HH:mm"))
    .withColumn("termination_date", F.to_date("termination_date", "dd-MM-yyyy HH:mm"))

    #Active/Inactive
    .withColumn("employee_status", F.when(F.col("termination_date").isNull(), "ACTIVE") .otherwise("INACTIVE"))
    # Data quality
    .dropDuplicates(["employee_id"])
    .filter(F.col("employee_id").isNotNull())
    .filter(F.col("email").isNotNull())
)

# Write to Silver
(df_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("employee")
)

print("Silver employee created")


# to display the data from the table silver employee
display(spark.read.table("plworkforce_catalog.002_silver.employee"))